# 01 — Build Dataset
Converts raw CUAD contracts from HuggingFace into a sentence-level dataset with NER tags and classifier labels.
**Run this notebook once before running 02_train_ner or 03_train_classifier.**

## Cell 1 — Imports

In [ ]:
import os, re, json, random
from collections import Counter
from datasets import load_dataset, Dataset, DatasetDict

os.environ['GIT_PYTHON_REFRESH'] = 'quiet'
print('Imports OK')

## Cell 2 — Constants & Label Schema

In [ ]:
DATASET_ID = 'tanvitakavane/datanauts_project_cuad-deadline-ner-version2'
SAVE_PATH  = '../data/deadline_sentences'

NER_LABELS = ['O','B-EXP_DATE','I-EXP_DATE','B-START_DATE','I-START_DATE','B-DURATION','I-DURATION']
NER_L2I    = {l: i for i, l in enumerate(NER_LABELS)}

CLF_L2I = {'none': 0, 'expiration': 1, 'effective': 2, 'renewal': 3}
CLF_I2L = {0: 'none', 1: 'expiration', 2: 'effective', 3: 'renewal'}

MONTH_WORDS = {
    'january','february','march','april','may','june','july',
    'august','september','october','november','december',
    'jan','feb','mar','apr','jun','jul','aug','sep','oct','nov','dec',
}
DURATION_WORDS = {
    'year','years','month','months','day','days','week','weeks',
    'quarter','quarters','one','two','three','four','five','six',
    'seven','eight','nine','ten','twelve','thirty','sixty','ninety',
    'annual','annually',
}

print('NER labels :', NER_LABELS)
print('CLF labels :', CLF_I2L)

## Cell 3 — Helper Functions

In [ ]:
def clean_clause(raw):
    if not raw or str(raw) in ('[]', 'null', 'None'):
        return ''
    try:
        parsed = json.loads(str(raw).replace("'", '"'))
        return ' '.join(str(p) for p in parsed).strip() if isinstance(parsed, list) else str(parsed).strip()
    except Exception:
        return re.sub(r"^\[?['\"]?|['\"]?\]?$", '', str(raw).strip()).strip()

def sent_split(text):
    text = re.sub(r'\n{2,}', ' ', text)
    text = re.sub(r'\n', ' ', text)
    return [s.strip() for s in re.split(r'(?<=[.?!])\s+(?=[A-Z0-9\(])', text) if len(s.strip()) > 20]

def overlaps(sent, clause, threshold=0.5):
    if not clause:
        return False
    cw = set(clause.lower().split())
    sw = set(sent.lower().split())
    return len(cw & sw) / max(len(cw), 1) >= threshold

def bio_tag(tokens, ctype):
    tags = ['O'] * len(tokens)
    if ctype == 'none':
        return tags
    if ctype == 'expiration':
        b, i, check = 'B-EXP_DATE', 'I-EXP_DATE', lambda t: t.lower() in MONTH_WORDS or bool(re.fullmatch(r'\d{4}', t.rstrip('.,'))) 
    elif ctype == 'effective':
        b, i, check = 'B-START_DATE', 'I-START_DATE', lambda t: t.lower() in MONTH_WORDS or bool(re.fullmatch(r'\d{4}', t.rstrip(',.'))) 
    else:
        b, i, check = 'B-DURATION', 'I-DURATION', lambda t: t.lower().rstrip('.,;:') in DURATION_WORDS
    in_span = False
    for idx, tok in enumerate(tokens):
        if check(tok):
            tags[idx] = i if in_span else b
            in_span = True
        else:
            in_span = False
    return tags

print('Helper functions defined')

# Quick sanity-check on bio_tag
test_tokens = ['This','Agreement','expires','December','31,','2025']
test_tags   = bio_tag(test_tokens, 'expiration')
print('bio_tag test:', list(zip(test_tokens, test_tags)))

## Cell 4 — Load Raw Dataset from HuggingFace

In [ ]:
print(f'Loading {DATASET_ID} ...')
raw = load_dataset(DATASET_ID)
print(raw)
print(f'\nContracts — train: {len(raw["train"])} | val: {len(raw["val"])} | test: {len(raw["test"])}')
print('\nColumn names:', raw['train'].column_names)

## Cell 5 — Inspect a Sample Contract Row

In [ ]:
row = raw['train'][0]
print('Filename        :', row.get('Filename', ''))
print('event_type      :', row.get('event_type', ''))
print('Expiration Date :', str(row.get('Expiration Date', ''))[:120])
print('expiration_iso  :', row.get('expiration_date_iso', ''))
print('Effective Date  :', str(row.get('Effective Date', ''))[:120])
print('effective_iso   :', row.get('effective_date_iso', ''))
print('OCR text length :', len(row.get('ocr_text', '') or ''))
print('OCR preview     :', (row.get('ocr_text', '') or '')[:300])

## Cell 6 — Process One Contract (test before full build)

In [ ]:
def process_contract(row, split_name):
    exp_clause  = clean_clause(row.get('Expiration Date',  '') or '')
    exp_iso     = row.get('expiration_date_iso') or None
    eff_clause  = clean_clause(row.get('Effective Date',   '') or '')
    eff_iso     = row.get('effective_date_iso')  or None
    ren_clause  = clean_clause(row.get('Renewal Term',     '') or '')
    ocr         = row.get('ocr_text', '') or ''
    contract_id = row.get('Filename', '')
    event_type  = row.get('event_type', '') or ''
    examples    = []

    for sent in sent_split(ocr):
        tokens = sent.split()
        if len(tokens) < 5:
            continue
        if exp_clause and overlaps(sent, exp_clause):
            clf   = 1                                       # always "expiration" event type
            ctype = 'expiration' if exp_iso else 'duration'  # NER tags EXP_DATE if explicit, DURATION if computable
            gt    = exp_iso or ''
        elif eff_clause and overlaps(sent, eff_clause):
            clf, ctype, gt = 2, 'effective', (eff_iso or '')
        elif ren_clause and overlaps(sent, ren_clause):
            clf, ctype, gt = 3, 'duration', ''              # renewal → NER tags DURATION
        else:
            clf, ctype, gt = 0, 'none', ''

        ner_tags = [NER_L2I[t] for t in bio_tag(tokens, ctype)]
        examples.append({
            'contract_id': contract_id, 'split': split_name,
            'sentence': sent, 'tokens': tokens,
            'ner_tags': ner_tags, 'classifier_label': clf,
            'ground_truth_date': gt, 'event_type': event_type,
        })
    return examples

# Test on first contract
test_examples = process_contract(raw['train'][0], 'train')
print(f'Contract 0 produced {len(test_examples)} sentences')
print(f'Label distribution: {Counter(e["classifier_label"] for e in test_examples)}')

# Show sentences that have NER entities
for ex in test_examples:
    if ex['classifier_label'] > 0:
        print(f'\n  clf={CLF_I2L[ex["classifier_label"]]}  gt={ex["ground_truth_date"]}')
        print(f'  sentence: {ex["sentence"][:120]}')
        ner_str = list(zip(ex['tokens'][:12], [NER_LABELS[t] for t in ex['ner_tags'][:12]]))
        print(f'  ner_tags: {ner_str}')
        break

## Cell 7 — Build All Splits (train / val / test)

In [ ]:
def build_split(hf_split, name):
    rows = []
    for row in hf_split:
        rows.extend(process_contract(row, name))
    return Dataset.from_list(rows)

print('Building train split...')
train_ds = build_split(raw['train'], 'train')
print(f'  train: {len(train_ds)} sentences')

print('Building val split...')
val_ds = build_split(raw['val'], 'val')
print(f'  val  : {len(val_ds)} sentences')

print('Building test split...')
test_ds = build_split(raw['test'], 'test')
print(f'  test : {len(test_ds)} sentences')

## Cell 8 — Dataset Statistics

In [ ]:
print('\n=== Sentence-Level Dataset Statistics ===')
for name, ds in [('train', train_ds), ('val', val_ds), ('test', test_ds)]:
    c = Counter(ds['classifier_label'])
    total = len(ds)
    print(f'  [{name}]  {total:>6} sentences | '
          f'none={c[0]:>5} ({100*c[0]/total:.1f}%)  '
          f'expiration={c[1]:>4} ({100*c[1]/total:.1f}%)  '
          f'effective={c[2]:>4} ({100*c[2]/total:.1f}%)  '
          f'renewal={c[3]:>4} ({100*c[3]/total:.1f}%)')

    ner_counts = Counter()
    for tags in ds['ner_tags']:
        ner_counts.update(tags)
    print(f'        NER tag distribution: { {NER_LABELS[k]: v for k,v in sorted(ner_counts.items())} }')

## Cell 9 — Inspect Sample Sentences with Labels

In [ ]:
print('--- Sample EXPIRATION sentences ---')
shown = 0
for ex in train_ds:
    if ex['classifier_label'] == 1 and shown < 3:
        print(f'  sentence : {ex["sentence"][:120]}')
        tokens_tags = list(zip(ex['tokens'][:15], [NER_LABELS[t] for t in ex['ner_tags'][:15]]))
        print(f'  ner_tags : {tokens_tags}')
        print(f'  gt_date  : {ex["ground_truth_date"]}\n')
        shown += 1

print('--- Sample EFFECTIVE sentences ---')
shown = 0
for ex in train_ds:
    if ex['classifier_label'] == 2 and shown < 3:
        print(f'  sentence : {ex["sentence"][:120]}')
        tokens_tags = list(zip(ex['tokens'][:15], [NER_LABELS[t] for t in ex['ner_tags'][:15]]))
        print(f'  ner_tags : {tokens_tags}')
        print(f'  gt_date  : {ex["ground_truth_date"]}\n')
        shown += 1

print('--- Sample RENEWAL sentences ---')
shown = 0
for ex in train_ds:
    if ex['classifier_label'] == 3 and shown < 3:
        print(f'  sentence : {ex["sentence"][:120]}')
        tokens_tags = list(zip(ex['tokens'][:15], [NER_LABELS[t] for t in ex['ner_tags'][:15]]))
        print(f'  ner_tags : {tokens_tags}\n')
        shown += 1

## Cell 10 — Save Dataset to Disk

In [ ]:
dd = DatasetDict({'train': train_ds, 'val': val_ds, 'test': test_ds})

os.makedirs(SAVE_PATH, exist_ok=True)
dd.save_to_disk(SAVE_PATH)
print(f'Dataset saved to: {SAVE_PATH}')

# Save label maps
os.makedirs('../data', exist_ok=True)
with open('../data/label_maps.json', 'w') as f:
    json.dump({
        'ner_label2id': NER_L2I,
        'ner_id2label': {str(k): v for k, v in enumerate(NER_LABELS)},
        'clf_label2id': CLF_L2I,
        'clf_id2label': {str(k): v for k, v in CLF_I2L.items()},
        'save_path': SAVE_PATH,
    }, f, indent=2)
print('Label maps saved to: ../data/label_maps.json')
print('\nDone! Run 02_train_ner.ipynb or 03_train_classifier.ipynb next.')